# Customer Intelligence Platform — Notebook 0: Data Generation & EDA
**Portfolio Project | Notebook 1 of 5**

---

## Learning Objectives
1. Generate a realistic, behaviorally-structured synthetic customer dataset (recency/frequency/monetary + engagement + demographic features) with a latent segment structure and a churn label plausibly driven by those features
2. Practice structured exploratory data analysis: distribution checks, missing-value auditing, correlation/multicollinearity screening, and target-vs-feature relationships
3. Establish the raw → processed data contract this entire project will build on top of

> **Senior engineer framing:** Every downstream notebook (segmentation, dimensionality reduction, the churn pipeline, MLflow tracking) depends on the data produced here being **realistic enough to be non-trivial** — if the synthetic data is too clean, clustering "succeeds" trivially and the churn model hits 0.99 AUC without needing a good pipeline, and none of the harder lessons land. This notebook deliberately injects overlap between segments, missing values, and noisy churn labels so the rest of the project requires real engineering rigor, not just correct syntax.
>
> In a real job, you would never generate your own labels this way — this is purely a stand-in for a licensable dataset (e.g. the IBM Telco Customer Churn dataset, or the UCI Online Retail dataset for RFM analysis). The important thing is that the **feature schema and workflow below are realistic**, so everything you build transfers directly to a real dataset later — swapping the data source is a one-line change in this notebook, nothing downstream needs to change.

In [ ]:
# ── Imports & reproducibility ───────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

DATA_RAW_DIR = Path("../data/raw")
DATA_PROCESSED_DIR = Path("../data/processed")
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

---
## Part 1: Designing a Latent Segment Structure

Real customer bases aren't homogeneous — they're mixtures of distinct behavioral groups. We'll define **four latent segments**, each with its own distribution over every feature, then sample customers from a mixture of these segments. This is exactly the generative assumption K-Means implicitly makes (spherical Gaussian mixtures), which is precisely why K-Means will do reasonably well on this data in Notebook 1 — and precisely why it's important to *also* try DBSCAN and hierarchical clustering to confirm that assumption isn't hiding something.

| Segment | Profile | Expected churn risk |
|---|---|---|
| `champions` | Long tenure, high spend, very recent activity, low support burden, high satisfaction | Low |
| `steady_value` | Loyal but price-sensitive — moderate spend, heavy discount usage | Low–moderate |
| `at_risk` | Long gap since last purchase, high support tickets, low satisfaction | High |
| `new_onboarding` | Very short tenure, behavior not yet settled | Uncertain / moderate |

The `true_segment` label is kept **only** for later validating our unsupervised clustering results against a known answer (Notebook 1, optional section) — it is never used as a model feature, since in a real deployment this ground truth would not exist.

In [ ]:
# ── Per-segment generative parameters ────────────────────────────────────────
SEGMENT_WEIGHTS = {
    "champions": 0.25,
    "steady_value": 0.30,
    "at_risk": 0.25,
    "new_onboarding": 0.20,
}

N_CUSTOMERS = 6000


def generate_segment(name: str, n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Sample n customers from one latent segment's feature distributions."""
    if name == "champions":
        tenure = rng.normal(48, 12, n).clip(1)
        monthly_spend = rng.normal(180, 40, n).clip(5)
        recency_days = rng.exponential(5, n)
        frequency_12m = rng.poisson(18, n)
        support_tickets_12m = rng.poisson(1, n)
        discount_usage_rate = rng.beta(2, 8, n)
        avg_session_minutes = rng.normal(25, 6, n).clip(1)
        num_products = rng.poisson(3, n) + 1
        satisfaction_score = rng.normal(9, 0.8, n).clip(0, 10)
        contract_probs = {"month-to-month": 0.1, "one-year": 0.3, "two-year": 0.6}
    elif name == "steady_value":
        tenure = rng.normal(30, 10, n).clip(1)
        monthly_spend = rng.normal(70, 15, n).clip(5)
        recency_days = rng.exponential(10, n)
        frequency_12m = rng.poisson(10, n)
        support_tickets_12m = rng.poisson(2, n)
        discount_usage_rate = rng.beta(6, 4, n)
        avg_session_minutes = rng.normal(15, 5, n).clip(1)
        num_products = rng.poisson(1.5, n) + 1
        satisfaction_score = rng.normal(7, 1.0, n).clip(0, 10)
        contract_probs = {"month-to-month": 0.35, "one-year": 0.45, "two-year": 0.2}
    elif name == "at_risk":
        tenure = rng.normal(15, 8, n).clip(1)
        monthly_spend = rng.normal(35, 10, n).clip(5)
        recency_days = rng.exponential(45, n)
        frequency_12m = rng.poisson(3, n)
        support_tickets_12m = rng.poisson(4, n)
        discount_usage_rate = rng.beta(3, 5, n)
        avg_session_minutes = rng.normal(6, 3, n).clip(0.5)
        num_products = rng.poisson(1, n) + 1
        satisfaction_score = rng.normal(4, 1.2, n).clip(0, 10)
        contract_probs = {"month-to-month": 0.75, "one-year": 0.2, "two-year": 0.05}
    elif name == "new_onboarding":
        tenure = rng.normal(3, 2, n).clip(0)
        monthly_spend = rng.normal(50, 20, n).clip(5)
        recency_days = rng.exponential(8, n)
        frequency_12m = rng.poisson(4, n)
        support_tickets_12m = rng.poisson(2, n)
        discount_usage_rate = rng.beta(4, 4, n)
        avg_session_minutes = rng.normal(12, 5, n).clip(1)
        num_products = rng.poisson(1, n) + 1
        satisfaction_score = rng.normal(6.5, 1.3, n).clip(0, 10)
        contract_probs = {"month-to-month": 0.7, "one-year": 0.25, "two-year": 0.05}
    else:
        raise ValueError(f"Unknown segment: {name}")

    contract_type = rng.choice(list(contract_probs.keys()), size=n, p=list(contract_probs.values()))

    return pd.DataFrame({
        "tenure_months": tenure,
        "monthly_spend": monthly_spend,
        "recency_days": recency_days,
        "frequency_12m": frequency_12m,
        "support_tickets_12m": support_tickets_12m,
        "discount_usage_rate": discount_usage_rate,
        "avg_session_minutes": avg_session_minutes,
        "num_products": num_products,
        "satisfaction_score": satisfaction_score,
        "contract_type": contract_type,
        "true_segment": name,
    })


segment_frames = []
for segment_name, weight in SEGMENT_WEIGHTS.items():
    n_segment = int(round(N_CUSTOMERS * weight))
    segment_frames.append(generate_segment(segment_name, n_segment, rng))

df = pd.concat(segment_frames, ignore_index=True)

# Independent, segment-agnostic categorical attributes
df["region"] = rng.choice(["North", "South", "East", "West"], size=len(df))
df["acquisition_channel"] = rng.choice(
    ["organic", "paid_search", "referral", "social", "email"], size=len(df),
    p=[0.30, 0.25, 0.20, 0.15, 0.10],
)
df["payment_method"] = rng.choice(
    ["credit_card", "bank_transfer", "digital_wallet", "mailed_check"], size=len(df),
    p=[0.45, 0.25, 0.22, 0.08],
)

# Derived feature: lifetime spend, with some multiplicative noise (not a pure linear combination)
df["total_spend_lifetime"] = (df["tenure_months"] * df["monthly_spend"] * rng.normal(1.0, 0.08, len(df))).clip(0)

# Shuffle so segment order isn't trivially recoverable by row position
df = df.sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)
df.insert(0, "customer_id", [f"CUST-{i:05d}" for i in range(len(df))])

df.shape

### TODO 1 — Churn probability as a function of behavior, not just segment

Fill in `compute_churn_probability` below. It should combine **standardized** versions of `recency_days`, `satisfaction_score`, and `support_tickets_12m` into a logistic function, plus a small segment-level base-rate offset. This mirrors how you'd actually build a labeling heuristic (or validate a real label) against known churn drivers.

**HINT:**
- Standardize a feature with `(x - x.mean()) / x.std()` so features with different units/scales contribute comparably to the linear combination
- Higher `recency_days` (long time since last activity) → **higher** churn probability → positive weight
- Higher `satisfaction_score` → **lower** churn probability → negative weight
- Higher `support_tickets_12m` → **higher** churn probability → positive weight
- Squash the linear combination into a probability with the sigmoid function: $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- Add `rng.normal(0, 0.4, len(df))` to the linear combination *before* the sigmoid so the label isn't perfectly deterministic from the features (real churn labels always have noise the features don't fully explain)
- Include a small negative **intercept** (try `-1.3`) in `z` — without one, a mixture of standardized features centered at 0 plus segment offsets averaging near 0 will sigmoid out to roughly 45-50% churn, which is unrealistically high for most subscription businesses (aim for ~20-30% overall)


In [ ]:
def compute_churn_probability(frame: pd.DataFrame, rng: np.random.Generator) -> np.ndarray:
    """Return a per-customer churn probability in [0, 1], driven by recency,
    satisfaction, and support-ticket volume, plus a segment base rate and noise.
    """
    # TODO: standardize the three driver columns
    recency_z = ...      # (frame["recency_days"] - frame["recency_days"].mean()) / frame["recency_days"].std()
    satisfaction_z = ...
    tickets_z = ...

    segment_base_rate = frame["true_segment"].map({
        "champions": -1.5,
        "steady_value": -0.5,
        "at_risk": 1.5,
        "new_onboarding": 0.0,
    }).to_numpy()

    noise = rng.normal(0, 0.4, len(frame))

    # TODO: combine into a linear score `z`, weighting recency_z and tickets_z positively,
    # satisfaction_z negatively, then add segment_base_rate and noise
    z = ...

    # TODO: apply the sigmoid function to turn z into a probability
    probability = ...
    return probability


churn_probability = compute_churn_probability(df, rng)
df["churned"] = rng.binomial(1, churn_probability)
df["churned"].mean()

### TODO 2 — Inject realistic missingness

Real customer tables always have gaps — a satisfaction survey nobody answered, a session-tracking pixel that failed to fire. Inject **missing-completely-at-random** values into `satisfaction_score` (~4% missing) and `avg_session_minutes` (~6% missing).

**HINT:** build a boolean mask with `rng.random(len(df)) < missing_rate`, then use `df.loc[mask, column] = np.nan`.

In [ ]:
# TODO: inject ~4% missing into "satisfaction_score" and ~6% missing into "avg_session_minutes"
satisfaction_missing_mask = ...
avg_session_missing_mask = ...

df.loc[satisfaction_missing_mask, "satisfaction_score"] = np.nan
df.loc[avg_session_missing_mask, "avg_session_minutes"] = np.nan

df.isna().sum()

---
## Part 2: Exploratory Data Analysis

With the dataset assembled, work through the same EDA checklist you'd apply to *any* new tabular dataset before modeling: shape/dtypes, missingness, distributions, correlation structure, and target relationships.

In [ ]:
print(df.shape)
df.info()
df.describe(include="number").T

### TODO 3 — Distribution grid

Plot histograms (with `sns.histplot`) for `tenure_months`, `monthly_spend`, `recency_days`, `frequency_12m`, `support_tickets_12m`, and `satisfaction_score` in a 2×3 grid of subplots. Look for multi-modality — a visible sign of the underlying segment mixture, even before we run any clustering algorithm.

**HINT:** `fig, axes = plt.subplots(2, 3, figsize=(15, 8))`, then loop over `zip(axes.flat, columns)`.

In [ ]:
numeric_cols_to_plot = [
    "tenure_months", "monthly_spend", "recency_days",
    "frequency_12m", "support_tickets_12m", "satisfaction_score",
]

# TODO: build the 2x3 histogram grid described above
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols_to_plot):
    pass  # TODO: sns.histplot(df[col].dropna(), kde=True, ax=ax); ax.set_title(col)
plt.tight_layout()
plt.show()

### TODO 4 — Correlation and multicollinearity screen

Compute the correlation matrix of the numeric features and render it as a `sns.heatmap`. Identify any pair with $|r| > 0.7$ — this matters directly for Notebook 3, since highly-correlated features can destabilize linear-model coefficients and inflate PCA's first component with redundant signal.

**HINT:** `df[numeric_cols].corr()`, then `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)`.

In [ ]:
numeric_cols = [
    "tenure_months", "monthly_spend", "total_spend_lifetime", "recency_days",
    "frequency_12m", "support_tickets_12m", "discount_usage_rate",
    "avg_session_minutes", "num_products", "satisfaction_score",
]

# TODO: compute df[numeric_cols].corr() and plot it as a heatmap
corr = ...
plt.figure(figsize=(9, 7))
# sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

### TODO 5 — Churn rate by categorical attribute

For `contract_type` and `acquisition_channel`, compute the churn rate (`groupby(col)["churned"].mean()`) and plot as a bar chart. You should see `month-to-month` and `at_risk`-heavy channels show a materially higher churn rate — this is your first quantitative signal that these features will matter to the Notebook 3 classifier.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# TODO: churn_by_contract = df.groupby("contract_type")["churned"].mean().sort_values()
# TODO: churn_by_channel = df.groupby("acquisition_channel")["churned"].mean().sort_values()
churn_by_contract = ...
churn_by_channel = ...

# churn_by_contract.plot(kind="barh", ax=axes[0], title="Churn rate by contract type")
# churn_by_channel.plot(kind="barh", ax=axes[1], title="Churn rate by acquisition channel")
plt.tight_layout()
plt.show()

---
## Part 3: Persisting the Raw → Processed Data Contract

- **`data/raw/customers_raw.csv`** — the full generated dataset, including `true_segment` (the "answer key" for later cluster validation) and missing values, exactly as it "arrived". Nothing downstream should ever mutate this file.
- **`data/processed/customers.csv`** — the modeling-ready table: `true_segment` removed (it must never leak into features), missing values **still present** (imputation is a `Pipeline` step, not a one-off notebook operation — see Notebook 3).
- **`data/processed/ground_truth_segments.csv`** — `customer_id` + `true_segment` only, kept *separately* so Notebook 1 can optionally score clustering quality against it without that information ever touching the feature table.

In [ ]:
df.to_csv(DATA_RAW_DIR / "customers_raw.csv", index=False)

processed_df = df.drop(columns=["true_segment"])
processed_df.to_csv(DATA_PROCESSED_DIR / "customers.csv", index=False)

df[["customer_id", "true_segment"]].to_csv(DATA_PROCESSED_DIR / "ground_truth_segments.csv", index=False)

print("Saved:")
print(f"  {DATA_RAW_DIR / 'customers_raw.csv'}  ({df.shape[0]} rows, {df.shape[1]} cols)")
print(f"  {DATA_PROCESSED_DIR / 'customers.csv'}  ({processed_df.shape[0]} rows, {processed_df.shape[1]} cols)")
print(f"  {DATA_PROCESSED_DIR / 'ground_truth_segments.csv'}")

---
## Senior Engineer Notes & Best Practices

1. **Never let a "ground truth" or label-adjacent column silently ride along into a feature table.** `true_segment` is deliberately persisted to a *separate* file — a shared column name between a features file and an answer-key file is exactly the kind of thing that causes silent leakage six months later when someone joins tables without checking.
2. **Raw is immutable.** Every notebook after this one reads from `data/processed/`, never `data/raw/` — if the cleaning logic ever needs to change, you rerun this notebook, you don't hand-edit an intermediate file.
3. **Missingness should be introduced (or discovered) and documented explicitly**, not silently dropped. Dropping rows with `dropna()` at the EDA stage is one of the most common ways a class-imbalanced target gets even more imbalanced without anyone noticing.
4. **EDA before modeling is not optional busywork.** The correlation screen and churn-by-category breakdown you just ran directly predict which features will matter in Notebook 3 and which pairs might destabilize a linear model — you're building intuition you'll need to sanity-check the pipeline's results later.
5. **Synthetic data is a teaching tool, not a substitute for validating on real data eventually.** Keep the schema realistic so the leap to a real dataset (e.g. swapping in the IBM Telco Churn dataset) requires changing only this notebook.

## Key Takeaways
- A believable customer dataset needs a latent mixture structure, feature-driven (not purely random) labels, realistic missingness, and label noise — otherwise downstream modeling lessons don't land
- `data/raw/` (immutable source) → `data/processed/` (modeling-ready, but *not* imputed — that belongs in a `Pipeline`) is the contract every later notebook depends on
- Ground-truth/ID columns that could leak must be physically separated from the feature table, not just "remembered to be excluded"